<a href="https://colab.research.google.com/github/uznainrashid/DataScience/blob/main/Titanic_Advanced_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

# Data load karein (Apna sahi path ya URL yahan likhein)
full_data = pd.read_csv('/content/titanic.csv')

In [2]:
# 1. Family Survival Rate Feature Engineering
full_data['Surname'] = full_data['Name'].str.split(',').str[0]
full_data['FamilySize'] = full_data['SibSp'] + full_data['Parch'] + 1
full_data['FamilyID'] = full_data['Surname'] + '_' + full_data['FamilySize'].astype(str)

total_family_survived = full_data.groupby('FamilyID')['Survived'].transform('sum')
full_data['Family_Survival_Rate'] = (total_family_survived - full_data['Survived']) / (full_data['FamilySize'] - 1)
full_data.loc[full_data['FamilySize'] == 1, 'Family_Survival_Rate'] = 0.5

# 2. Title-Based Missing Age Imputation
full_data['Title'] = full_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
full_data['Age'] = full_data['Age'].fillna(full_data.groupby('Title')['Age'].transform('median'))

# 3. Drop Unnecessary Columns
full_data.drop(['Name', 'Surname', 'FamilyID', 'Ticket', 'Cabin', 'Title'], axis=1, inplace=True, errors='ignore')

# 4. One-Hot Encoding for Categorical Columns
gender_dummies = pd.get_dummies(full_data['Sex'], drop_first=True, dtype=int)
embarked_dummies = pd.get_dummies(full_data['Embarked'], drop_first=True, dtype=int)
full_data = pd.concat([full_data, gender_dummies, embarked_dummies], axis=1)
full_data.drop(['Sex', 'Embarked'], axis=1, inplace=True, errors='ignore')

In [3]:
# Inputs aur Target alag karna
X = full_data.drop(['Survived', 'PassengerId'], axis=1)
y = full_data['Survived']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Fit karna
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy Check
y_pred = model.predict(X_test)
print(f"Final Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Final Model Accuracy: 86.03%


In [4]:
# Trained model ko pickle file mein save karna
with open('titanic_model.pkl', 'wb') as file:
    pickle.dump(model, file)

print("Model successfully saved as 'titanic_model.pkl'!")

Model successfully saved as 'titanic_model.pkl'!
